# Week 2: basic image operations

**What we do.** We apply simple pixel operations to a photograph: brightening and darkening, changing the contrast, gamma correction, binarization, mirroring and cropping with masks.

**What to expect.** Every operation here acts on each pixel separately (or rearranges pixels), so it is easy to predict its effect from a graph of the map `old intensity → new intensity`. We will see such graphs next to the images.

**How to read the code.** All functions come from `wavelets_lab.images`; they take an image (an array on `[0, 255]`) and return a new image, never modifying the input. The last section of the notebook describes every function in detail.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import images

## The test image

**What we do.** We load the `camera` photograph used in this notebook.

**What to expect.** A 512×512 greyscale image with values from 0 to 255.

**How to read the code.** `images.load_test_image` returns a `float64` array; we keep it in the variable `camera`.

In [ ]:
camera = images.load_test_image("camera")
camera.shape, camera.min(), camera.max()

## Intensity maps

**What we do.** Before looking at images we draw the functions that the operations apply to every pixel value `x ∈ [0, 255]`.

**What to expect.** Brightening `1.2 x` is a steeper line that saturates (is clipped) at 255; darkening `x / 1.2` is a flatter line; `25 + 0.8 x` lifts black to 25 and white to 229, so the output range shrinks (lower contrast); gamma correction with `γ = 0.5` is a concave curve that stretches dark tones. A slope larger than 1 increases contrast in that range of intensities, a slope smaller than 1 decreases it.

**How to read the code.** `x = np.linspace(0, 255, 256)` is a grid of all intensities. We apply each function from `images` to this 1D array (they work for arrays of any shape) and plot the result against `x`.

In [ ]:
x = np.linspace(0.0, 255.0, 256)
maps = {
    "brighter: 1.2 x": images.scale_intensity(x, 1.2),
    "darker: x / 1.2": images.scale_intensity(x, 1 / 1.2),
    "lower contrast: 25 + 0.8 x": images.affine_intensity(x, 0.8, 25),
    "gamma 0.5": images.gamma_correction(x, 0.5),
    "gamma 2": images.gamma_correction(x, 2.0),
}
plt.figure(figsize=(6, 6))
plt.plot(x, x, "k--", label="identity")
for label, y in maps.items():
    plt.plot(x, y, label=label)
plt.xlabel("input intensity")
plt.ylabel("output intensity")
plt.legend()
plt.show()

## Brightness and contrast

**What we do.** We apply the operations to the photograph.

**What to expect.** Brightening makes the sky white everywhere (saturation, detail is lost); lowering the contrast gives a greyish, "foggy" image; `stretch_contrast` maps the darkest pixel to 0 and the brightest to 255 and restores the contrast of the foggy image; `γ = 0.5` reveals detail in the dark coat.

**How to read the code.** The dictionary `versions` maps titles to images. `plt.subplots(2, 3)` makes a 2×3 grid of panels and `axes.flat` goes through them in reading order; `zip(..., strict=True)` pairs panels with images and checks that their numbers agree.

In [ ]:
lower = images.affine_intensity(camera, 0.8, 25)
versions = {
    "original": camera,
    "brighter (x 1.2)": images.scale_intensity(camera, 1.2),
    "darker (/ 1.2)": images.scale_intensity(camera, 1 / 1.2),
    "lower contrast": lower,
    "lower contrast, stretched": images.stretch_contrast(lower),
    "gamma 0.5": images.gamma_correction(camera, 0.5),
}
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (title, image) in zip(axes.flat, versions.items(), strict=True):
    images.show_image(image, title, ax=ax)
plt.tight_layout()
plt.show()

## Histograms

**What we do.** We compare the histograms (how many pixels have each intensity) of the original, the low-contrast and the stretched image.

**What to expect.** Lowering the contrast squeezes the histogram into `[25, 229]`; stretching spreads it again over `[0, 255]`, so its shape is the same as the original one (the operations are affine).

**How to read the code.** `plt.hist(values, bins=64, range=(0, 255))` counts the values in 64 equal intervals. `.ravel()` turns the 2D image into a 1D list of pixels; `histtype="step"` draws outlines so that the histograms do not hide each other.

In [ ]:
plt.figure(figsize=(10, 4))
for title in ("original", "lower contrast", "lower contrast, stretched"):
    plt.hist(
        versions[title].ravel(), bins=64, range=(0, 255), histtype="step", label=title
    )
plt.xlabel("intensity")
plt.ylabel("number of pixels")
plt.legend()
plt.show()

## Binarization

**What we do.** We turn the photograph into black-and-white images with four thresholds.

**What to expect.** With a low threshold (50) almost everything except the darkest parts becomes white; with a high threshold (200) only the brightest parts (the sky) stay white. Binarization is the simplest segmentation: it separates "object" from "background" when their intensities differ.

**How to read the code.** `images.binarize(camera, threshold)` returns an image containing only 0 and 255; the f-string `f"threshold {threshold}"` puts the number into the title.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, threshold in zip(axes, [50, 100, 150, 200], strict=True):
    images.show_image(
        images.binarize(camera, threshold), f"threshold {threshold}", ax=ax
    )
plt.tight_layout()
plt.show()

## Mirroring

**What we do.** We extend the image to the right by its mirror image.

**What to expect.** A 512×1024 image whose right half is the left half flipped; the column axis now goes up to 1023. Mirror extension is how signals are usually extended at their boundary before filtering, because it creates no jump.

**How to read the code.** `images.mirror_extend(camera)` mirrors along columns (`axis=1`); the shape is printed in the title.

In [ ]:
extended = images.mirror_extend(camera)
fig, ax = plt.subplots(figsize=(12, 6))
images.show_image(extended, f"mirror extension, shape {extended.shape}", ax=ax)
plt.show()

## Cropping with masks

**What we do.** We cut out a rectangle (rows 60–299, columns 150–399) and replace the rest with a white background, once with a hard mask and once with a smooth mask whose weight falls to 0 over 40 pixels.

**What to expect.** The hard crop has sharp artificial edges; the smooth crop fades into the background. The profile plot shows the two masks along row 180: a step and a trapezoid.

**How to read the code.** `images.rectangle_mask(shape, top_left, bottom_right, ramp)` returns weights in `[0, 1]`; `images.apply_mask(image, mask)` computes `mask * image + (1 - mask) * 255`. The third panel is an ordinary graph, so it is labelled with `set_xlabel` and `set_ylabel`.

In [ ]:
hard = images.rectangle_mask(camera.shape, (60, 150), (300, 400))
smooth = images.rectangle_mask(camera.shape, (60, 150), (300, 400), ramp=40)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
images.show_image(images.apply_mask(camera, hard), "hard crop", ax=axes[0])
images.show_image(images.apply_mask(camera, smooth), "smooth crop", ax=axes[1])
axes[2].plot(hard[180], label="hard mask")
axes[2].plot(smooth[180], label="smooth mask")
axes[2].set_xlabel("column [pixel]")
axes[2].set_ylabel("mask weight")
axes[2].set_title("masks along row 180")
axes[2].legend()
plt.tight_layout()
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above, grouped by the file it comes from, with its purpose, input, output and a step-by-step recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.images` (`src/wavelets_lab/images.py`). Functions marked *used internally* are called by the others.

**How to read the code.** Signatures are written as in Python, `name(argument, optional=default)`; "array-like" means anything `np.asarray` accepts.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.linspace(a, b, n)` | `n` equally spaced numbers from `a` to `b` | numbers → 1D array |
| `plt.plot(x, y, style, label=...)` | draws a curve; `"k--"` = black dashed | arrays → line |
| `plt.hist(values, bins, range, histtype)` | histogram of values | 1D array → bars or outline |
| `array.ravel()` | flattens an array to 1D | array → 1D array |
| `plt.subplots`, `ax.set_xlabel`, `ax.set_ylabel`, `ax.legend` | figure layout and labels | – |

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.

#### `images.scale_intensity(image, factor)`

**What it is.** Multiplies every pixel by `factor` and clips to `[0, 255]`. `factor > 1` brightens, `factor < 1` darkens.

**Input.**
- `image` (array-like): image on `[0, 255]`.
- `factor` (`float`): e.g. `1.2` or `1 / 1.2`.

**Output.** `float64` image on `[0, 255]`.

**How to write it yourself.**
1. Convert: `np.asarray(image, dtype=np.float64)`.
2. Multiply by `factor`.
3. Clip with `np.clip(result, 0.0, 255.0)` (internal helper `_clip`): bright pixels saturate at 255, which is why brightening loses detail in light areas.

#### `images.affine_intensity(image, gain, offset)`

**What it is.** The affine map `offset + gain * image`, clipped to `[0, 255]`. With `gain < 1` the range of intensities shrinks, i.e. the contrast is reduced; `offset` shifts the brightness.

**Input.**
- `image` (array-like): image on `[0, 255]`.
- `gain` (`float`): multiplicative coefficient, e.g. `0.8`.
- `offset` (`float`): additive coefficient, e.g. `25`.

**Output.** `float64` image on `[0, 255]`.

**How to write it yourself.**
1. Convert the image to `float64`.
2. Compute `offset + gain * image`.
3. Clip to `[0, 255]` with `np.clip` and return.

#### `images.normalize(image)` (used internally)

**What it is.** Linear rescaling of an image to `[0, 1]`: `(x - min) / (max - min)`.

**Input.**
- `image` (array-like): any image.

**Output.** `float64` array with minimum 0 and maximum 1; a constant image (where `max == min`) is mapped to zeros to avoid division by zero.

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`; `low, high = array.min(), array.max()`.
2. If `high == low`: return `np.zeros_like(array)`.
3. Otherwise return `(array - low) / (high - low)`.

#### `images.stretch_contrast(image)`

**What it is.** Maximises the contrast: the darkest pixel becomes 0 and the brightest 255, linearly in between. If the image already uses the full range, nothing changes.

**Input.**
- `image` (array-like): any image.

**Output.** `float64` image with minimum 0 and maximum 255.

**How to write it yourself.**
1. Rescale to `[0, 1]` with `normalize(image)`.
2. Multiply by 255 and return.

#### `images.gamma_correction(image, gamma)`

**What it is.** The non-linear map `255 * (x / 255) ** gamma`. It keeps 0 and 255 fixed; `gamma < 1` brightens dark tones (more contrast in the shadows), `gamma > 1` darkens (more contrast in the highlights).

**Input.**
- `image` (array-like): image on `[0, 255]`.
- `gamma` (`float`, positive).

**Output.** `float64` image on `[0, 255]`. Raises `ValueError` if `gamma <= 0`.

**How to write it yourself.**
1. If `gamma <= 0`, raise `ValueError("gamma must be positive")`.
2. Convert to `float64`, clip to `[0, 255]` and divide by 255 to get values in `[0, 1]`.
3. Raise to the power `gamma` (`unit ** gamma`) and multiply by 255.

#### `images.binarize(image, threshold)`

**What it is.** Thresholding into a black-and-white image: pixels brighter than `threshold` become 255, all others 0.

**Input.**
- `image` (array-like): image on `[0, 255]`.
- `threshold` (`float`): e.g. 50, 100, 150 or 200.

**Output.** `float64` array containing only the values 0 and 255.

**How to write it yourself.**
1. Compare elementwise: `np.asarray(image) > threshold` gives a Boolean array.
2. Use `np.where(condition, 255.0, 0.0)` to turn it into intensities.

#### `images.mirror_extend(image, axis=1)`

**What it is.** Appends the mirror image along an axis, e.g. a 512×512 image becomes 512×1024 with the right half being the left half flipped. Mirror extension is also the standard way of extending signals at their boundary before filtering.

**Input.**
- `image` (array-like): image.
- `axis` (`int`, default 1): 1 mirrors left-right (columns), 0 mirrors top-bottom (rows).

**Output.** `float64` array twice as large along `axis`.

**How to write it yourself.**
1. Convert to `float64`.
2. Flip along the axis: `np.flip(array, axis=axis)`.
3. Glue original and flipped copy: `np.concatenate([array, flipped], axis=axis)`.

#### `images.rectangle_mask(shape, top_left, bottom_right, ramp=0)`

**What it is.** Builds a mask (an array of weights in `[0, 1]`) equal to 1 inside a rectangle. With `ramp=0` the mask jumps to 0 outside (hard crop); with `ramp > 0` it decreases linearly to 0 over `ramp` pixels (smooth crop, no sharp artificial edge).

**Input.**
- `shape` (`tuple[int, int]`): `(rows, cols)` of the mask.
- `top_left` (`tuple[int, int]`): row and column of the first pixel inside.
- `bottom_right` (`tuple[int, int]`): row and column one past the last pixel inside (Python slicing convention).
- `ramp` (`int`, default 0): width of the transition in pixels.

**Output.** `float64` array of shape `shape` with values in `[0, 1]`.

**How to write it yourself.**
1. Create index grids that broadcast: `rows = np.arange(shape[0])[:, np.newaxis]`, `cols = np.arange(shape[1])[np.newaxis, :]`.
2. Distance (in pixels) of each row from the rectangle's row range: `row_dist = max(top - row, row - (bottom - 1), 0)` computed with `np.maximum` twice; the same for columns.
3. The distance of a pixel from the rectangle (in the max-norm) is `np.maximum(row_dist, col_dist)`; it is 0 inside.
4. Hard mask (`ramp <= 0`): return `(distance == 0)` as `float64`.
5. Smooth mask: return `np.clip(1 - distance / ramp, 0, 1)`, which is 1 inside, decreases linearly and reaches 0 at distance `ramp`.

#### `images.apply_mask(image, mask, background=255.0)`

**What it is.** Blends an image with a constant background: `mask * image + (1 - mask) * background`. Where the mask is 1 the image is kept, where it is 0 only the background (white by default) remains, and in between the two are mixed.

**Input.**
- `image` (array-like): greyscale image.
- `mask` (array-like): weights in `[0, 1]` of the same shape, e.g. from `rectangle_mask`.
- `background` (`float`, default 255): background intensity.

**Output.** `float64` image.

**How to write it yourself.**
1. Convert `mask` and `image` to `float64` arrays.
2. Return `weights * image + (1 - weights) * background`.